# Initial Data Cleaning - School Dataset
### Notebook Done By: Kai/Su

This notebook contains some of the processes for the Initial Data Cleaning phase, specifically for cleaning the school dataset. This notebook covers the process of cleaning the school dataset.

In [1]:
# Importing libraries

import ezodf
import pandas as pd
import numpy as np

In [2]:
# Opening the school dataset

file_path = "../../data/raw/five-year-ofsted-inspection-data_state-funded-schools.ods"
sheet_name = "Provider_level_data"

doc = ezodf.opendoc(file_path)
sheet = doc.sheets[sheet_name]

In [3]:
# Loading the school dataset into a Pandas DataFrame

rows = list(sheet.rows())
data = [[cell.value for cell in row] for row in rows]

df_raw = pd.DataFrame(data)
print(df_raw.head(10))

                                                  0   \
0  Five-Year Ofsted Inspection Data: State-funded...   
1  This worksheet contains one table. Each row in...   
2                                                URN   
3                                           100000.0   
4                                           100005.0   
5                                           100006.0   
6                                           100007.0   
7                                           100008.0   
8                                           100009.0   
9                                           100010.0   

                                   1   \
0                                None   
1                                None   
2                                Name   
3                  The Aldgate School   
4                 Thomas Coram Centre   
5                        Heath School   
6  Camden Primary Pupil Referral Unit   
7               Argyle Primary School   
8       West H

In [4]:
# Adding column names to the school dataset

headers = df_raw.iloc[3]
df = df_raw.iloc[4:].reset_index(drop=True)
df.columns = headers

print(df.head())
print(df.shape)

3  100000.0                  The Aldgate School  2025-04-02T00:00:00  OS  \
0  100005.0                 Thomas Coram Centre  2025-04-02T00:00:00  OS   
1  100006.0                        Heath School  2025-04-02T00:00:00  OS   
2  100007.0  Camden Primary Pupil Referral Unit  2025-04-02T00:00:00  OS   
3  100008.0               Argyle Primary School  2025-04-02T00:00:00  OS   
4  100009.0       West Hampstead Primary School  2025-04-02T00:00:00  OS   

3  State-funded schools  2024-12-31T00:00:00  2024-07-11T00:00:00  London  \
0  State-funded schools  2024-12-31T00:00:00  2014-07-02T00:00:00  London   
1  State-funded schools  2024-12-31T00:00:00  2024-07-04T00:00:00  London   
2  State-funded schools  2024-12-31T00:00:00  2015-05-05T00:00:00  London   
3  State-funded schools  2024-12-31T00:00:00  2022-11-17T00:00:00  London   
4  State-funded schools  2024-12-31T00:00:00  2012-09-10T00:00:00  London   

3 City of London Cities of London and Westminster  ...  None  1.0  1.0  1.0  \
0

In [5]:
# Standardising column names

def clean_colname(x):
    x = str(x).strip().lower()
    x = x.replace(" ", "_")
    x = "".join(ch for ch in x if ch.isalnum() or ch == "_")
    return x

df.columns = [clean_colname(c) for c in df.columns]
print(df.columns)

Index(['1000000', 'the_aldgate_school', '20250402t000000', 'os',
       'statefunded_schools', '20241231t000000', '20240711t000000', 'london',
       'city_of_london', 'cities_of_london_and_westminster', 'ec3a_5de',
       'local_authority_maintained', 'voluntary_aided_school', 'primary',
       'deprived', '10', 'none', '10', '10', '10', '10', 'yes', '20', '90',
       'yes', 'none'],
      dtype='object')


In [6]:
# Cleaning up columns in the school dataset

df = df.loc[:, ~df.columns.str.fullmatch("none")]
df = df.dropna(axis=1, how="all")

print(df.shape)

(131928, 24)


In [7]:
# Standardising Yes/No values into 1/0 values

def fix_yes_no(v):
    if isinstance(v, str):
        vv = v.strip().lower()
        if vv in ["yes", "y", "true"]:
            return 1
        if vv in ["no", "n", "false"]:
            return 0
    return v

df = df.applymap(fix_yes_no)

C:\Users\Nitro\AppData\Local\Temp\ipykernel_71084\3538245062.py:12: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df.applymap(fix_yes_no)


In [8]:
# Converting date columns into datetime

for col in df.columns:
    if "date" in col:
        df[col] = pd.to_datetime(df[col], errors="coerce")

print(df.dtypes)

1000000                             float64
the_aldgate_school                   object
20250402t000000                      object
os                                   object
statefunded_schools                  object
20241231t000000                      object
20240711t000000                      object
london                               object
city_of_london                       object
cities_of_london_and_westminster     object
ec3a_5de                             object
local_authority_maintained           object
voluntary_aided_school               object
primary                              object
deprived                             object
10                                   object
10                                  float64
10                                  float64
10                                  float64
10                                  float64
yes                                 float64
20                                  float64
90                              

In [9]:
# Checking for duplicate columns

print(df.columns[df.columns.duplicated()])

Index(['10', '10', '10', '10', 'yes'], dtype='object')


In [10]:
# Dropping duplicate columns and keeping the first one

df = df.loc[:, ~df.columns.duplicated()]

In [11]:
# Converting columns with numerical values into columns with numeric data types

def to_numeric_if_possible(s, threshold=0.8):
    if isinstance(s, pd.Series):
        converted = pd.to_numeric(s, errors="coerce")
        if converted.notna().mean() >= threshold:
            return converted
    return s

for col in df.columns:
    df[col] = to_numeric_if_possible(df[col])

print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 131928 entries, 0 to 131927
Data columns (total 19 columns):
 #   Column                            Non-Null Count   Dtype  
---  ------                            --------------   -----  
 0   1000000                           131927 non-null  float64
 1   the_aldgate_school                131927 non-null  object 
 2   20250402t000000                   131927 non-null  object 
 3   os                                131927 non-null  object 
 4   statefunded_schools               131927 non-null  object 
 5   20241231t000000                   131927 non-null  object 
 6   20240711t000000                   130757 non-null  object 
 7   london                            131927 non-null  object 
 8   city_of_london                    131927 non-null  object 
 9   cities_of_london_and_westminster  131927 non-null  object 
 10  ec3a_5de                          131927 non-null  object 
 11  local_authority_maintained        131927 non-null  o

In [12]:
# Dropping duplicate rows and filling missing values with the median

df = df.drop_duplicates()

num_cols = df.select_dtypes(include=[np.number]).columns
for col in num_cols:
    df[col] = df[col].fillna(df[col].median())

In [13]:
# Exporting the school dataset into a CSV file

df.to_csv("../../data/partially_processed/schools.csv", index=False)
print("Saved: schools.csv")

Saved: schools.csv


The Initial Data Cleaning phase for the school dataset has been completed, and the Initial Data Cleaning phase for the public transport stops dataset can begin.

The next notebook is ```notebooks/initial_data_cleaning/stop.ipynb```.